# Linear Algebra

That linear algebra is fun, is a widely accepted fact. This notebooks will guide you through some of the linear algebra fun you can realize with Heat. 

In [ ]:
from ipyparallel import Client
rc = Client(profile="default")
rc.ids

if len(rc.ids) == 0:
    print("No engines found")
else:
    print(f"{len(rc.ids)} engines found")

4 engines found


In [ ]:
%%px
import heat as ht

%px:   0%|          | 0/4 [00:00<?, ?tasks/s]

[stderr:1] /p/home/jusers/baumann7/jureca/.venv/lib/python3.13/site-packages/heat/core/_config.py:108: UserWarning: Heat has CUDA GPU-support (PyTorch version 2.9.1+cu128 and `torch.cuda.is_available() = True`), but CUDA-awareness of MPI could not be detected. This may lead to performance degradation as direct MPI-communication between GPUs is not possible.
  warnings.warn(
/p/software/default/stages/2026/software/zarr/3.1.5-gcccoreflexiblas-14.3.0-3.4.5/lib/python3.13/site-packages/google_crc32c/__init__.py:29: RuntimeWarning: As the c extension couldn't be imported, `google-crc32c` is using a pure python implementation that is significantly slower. If possible, please configure a c build environment and compile the extension
  warnings.warn(_SLOW_CRC32C_WARNING, RuntimeWarning)


[stderr:0] /p/home/jusers/baumann7/jureca/.venv/lib/python3.13/site-packages/heat/core/_config.py:108: UserWarning: Heat has CUDA GPU-support (PyTorch version 2.9.1+cu128 and `torch.cuda.is_available() = True`), but CUDA-awareness of MPI could not be detected. This may lead to performance degradation as direct MPI-communication between GPUs is not possible.
  warnings.warn(
/p/software/default/stages/2026/software/zarr/3.1.5-gcccoreflexiblas-14.3.0-3.4.5/lib/python3.13/site-packages/google_crc32c/__init__.py:29: RuntimeWarning: As the c extension couldn't be imported, `google-crc32c` is using a pure python implementation that is significantly slower. If possible, please configure a c build environment and compile the extension
  warnings.warn(_SLOW_CRC32C_WARNING, RuntimeWarning)


[stderr:3] /p/home/jusers/baumann7/jureca/.venv/lib/python3.13/site-packages/heat/core/_config.py:108: UserWarning: Heat has CUDA GPU-support (PyTorch version 2.9.1+cu128 and `torch.cuda.is_available() = True`), but CUDA-awareness of MPI could not be detected. This may lead to performance degradation as direct MPI-communication between GPUs is not possible.
  warnings.warn(
/p/software/default/stages/2026/software/zarr/3.1.5-gcccoreflexiblas-14.3.0-3.4.5/lib/python3.13/site-packages/google_crc32c/__init__.py:29: RuntimeWarning: As the c extension couldn't be imported, `google-crc32c` is using a pure python implementation that is significantly slower. If possible, please configure a c build environment and compile the extension
  warnings.warn(_SLOW_CRC32C_WARNING, RuntimeWarning)


[stderr:2] /p/home/jusers/baumann7/jureca/.venv/lib/python3.13/site-packages/heat/core/_config.py:108: UserWarning: Heat has CUDA GPU-support (PyTorch version 2.9.1+cu128 and `torch.cuda.is_available() = True`), but CUDA-awareness of MPI could not be detected. This may lead to performance degradation as direct MPI-communication between GPUs is not possible.
  warnings.warn(
/p/software/default/stages/2026/software/zarr/3.1.5-gcccoreflexiblas-14.3.0-3.4.5/lib/python3.13/site-packages/google_crc32c/__init__.py:29: RuntimeWarning: As the c extension couldn't be imported, `google-crc32c` is using a pure python implementation that is significantly slower. If possible, please configure a c build environment and compile the extension
  warnings.warn(_SLOW_CRC32C_WARNING, RuntimeWarning)


Before we dive into it, we specify the device we want to run this tutorial on.
If you have GPUs available, use `ht.use_device('gpu')`, otherwise, use `ht.use_device('cpu')`.
Both gives the same results, just at different performance.
For the small examples here, CPUs will do alright, so don't worry if this is all you have right now.

In [ ]:
%%px
ht.use_device('gpu')

## Matrix-Matrix Multiplication

The most basic operation in linear algebra is matrix-matrix multiplication ("matmul"). Doing it by hand for a small matrix is not difficult and in fact not very spectacular. However, in the distributed setting (e.g., on 4 GPUs) even such a simple operation is not trivial any more: just imagine you work together with 3 other people and each of you only knows one fourth of the columns of a matrix $A$ and one fourth of the rows of a matrix $B$. Together, you have to compute the product $AB$ such that in the end each of you only has one fourth of the columns of $AB$...

In [ ]:
%%px
A = ht.random.randn(10000, 10000, split=0)
B = ht.random.randn(A.shape[1], 10000, split=1)
C = ht.matmul(A, B)  # or A @ B
C

%px:   0%|          | 0/4 [00:00<?, ?tasks/s]

Out[1:4]: 
DNDarray([[ 8.3229e+01, -1.1637e+02, -9.9740e+01,  ...,  4.4171e+01,  4.1174e+01, -5.5997e+01],
          [ 6.4800e+01,  5.1846e+01, -3.2355e+01,  ..., -1.2001e+02, -8.1056e+01, -1.1048e+01],
          [ 2.1856e+02, -2.1662e+01,  5.0350e+01,  ...,  1.2031e+02,  5.2446e+01, -2.2736e+02],
          ...,
          [-1.9609e+02, -1.5745e+01, -8.8338e+01,  ..., -1.0855e+01, -1.4582e+02, -9.9323e+01],
          [ 6.7981e+01, -8.0185e+01, -5.2755e+01,  ..., -2.7894e+02,  7.7306e+01, -1.1127e+02],
          [ 1.0025e+00, -4.0758e+01, -9.3919e+00,  ...,  5.3513e+01, -1.2340e+02,  2.4453e+01]], dtype=ht.float32, device=gpu:0, split=0)

## QR Decomposition and Triangular Solve

Given a matrix $A$, its QR decomposition is given by $A=QR$ where $Q$ is an orthogonal matrix (i.e. its columns are pairwise orthonormal) and $R$ is an upper triangular matrix. 

Further information: [QR on Wikipedia](https://en.wikipedia.org/wiki/QR_decomposition)

In [ ]:
%%px
A = ht.random.randn(100000, 1000, split=0)
Q,R = ht.linalg.qr(A)
print(f'{Q.shape=} {Q.split=} {Q.device=} | {R.shape=} {R.split=} {R.device=}')

[stdout:1] Q.shape=(100000, 1000) Q.split=0 Q.device=device(gpu:0) | R.shape=(1000, 1000) R.split=None R.device=device(gpu:0)


[stdout:0] Q.shape=(100000, 1000) Q.split=0 Q.device=device(gpu:0) | R.shape=(1000, 1000) R.split=None R.device=device(gpu:0)


[stdout:2] Q.shape=(100000, 1000) Q.split=0 Q.device=device(gpu:0) | R.shape=(1000, 1000) R.split=None R.device=device(gpu:0)


[stdout:3] Q.shape=(100000, 1000) Q.split=0 Q.device=device(gpu:0) | R.shape=(1000, 1000) R.split=None R.device=device(gpu:0)


In [ ]:
%%px
print(Q.T @ Q)
print(f'Deviation of Q from orthonormality: {ht.linalg.norm(Q.T @ Q - ht.eye(Q.shape[1]), ord=ht.inf).item():.2e}')

[stdout:0] 
Deviation of Q from orthonormality: 3.46e-06


[stdout:2] 
Deviation of Q from orthonormality: 3.46e-06


[stdout:3] 
Deviation of Q from orthonormality: 3.46e-06


[stdout:1] DNDarray([[ 1.0000e+00, -4.1910e-09,  3.5798e-09,  ...,  5.6934e-10, -1.9209e-09, -1.8917e-09],
          [-4.1910e-09,  1.0000e+00,  3.7253e-09,  ...,  3.3178e-09, -4.4820e-09, -6.0536e-09],
          [ 3.5798e-09,  3.7253e-09,  1.0000e+00,  ...,  2.5029e-09,  1.5134e-09,  6.9849e-10],
          ...,
          [ 2.3283e-09, -1.1642e-09,  3.6671e-09,  ...,  1.0000e+00,  1.1642e-10, -6.4028e-10],
          [-1.5134e-09, -3.2596e-09,  2.3283e-10,  ...,  1.1642e-10,  1.0000e+00, -7.4506e-09],
          [-4.3656e-10, -6.5193e-09, -9.3132e-10,  ..., -6.4028e-10, -7.4506e-09,  1.0000e+00]], dtype=ht.float32, device=gpu:0, split=1)
Deviation of Q from orthonormality: 3.46e-06


In [ ]:
%%px
print(f'Datatype: {Q.dtype}, precision: {ht.types.finfo(Q.dtype).eps}')

[stdout:0] Datatype: <class 'heat.core.types.float32'>, precision: 1.1920928955078125e-07


[stdout:1] Datatype: <class 'heat.core.types.float32'>, precision: 1.1920928955078125e-07


[stdout:2] Datatype: <class 'heat.core.types.float32'>, precision: 1.1920928955078125e-07


[stdout:3] Datatype: <class 'heat.core.types.float32'>, precision: 1.1920928955078125e-07


While it may seem to users of numpy that Q deviates a lot from orthonormality, it's actually pretty close considering the precision of Heat's default datatypes.

In [ ]:
%%px
print(R)

[stdout:0] 


[stdout:1] DNDarray([[-3.1605e+02,  1.4788e+00,  2.4261e-01,  ..., -3.2345e-01, -2.4966e-01, -5.2142e-01],
          [ 0.0000e+00,  3.1645e+02,  6.8280e-01,  ..., -5.7349e-01,  1.6615e+00, -1.1413e+00],
          [ 0.0000e+00,  0.0000e+00, -3.1722e+02,  ...,  3.1893e-01, -1.1342e-01, -2.6120e-02],
          ...,
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  3.1597e+02,  7.4856e-01, -1.2496e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,  3.1432e+02, -1.0355e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,  0.0000e+00, -3.1442e+02]], dtype=ht.float32, device=gpu:0, split=None)


[stdout:2] 


[stdout:3] 


In [ ]:
%%px
norm_A = ht.linalg.norm(A)
print(f'Reconstruction error: {(ht.linalg.norm(Q@R - A)/norm_A).item()}')

[stdout:2] Reconstruction error: 2.6114551587852475e-07


[stdout:1] Reconstruction error: 2.6114551587852475e-07


[stdout:0] Reconstruction error: 2.6114551587852475e-07


[stdout:3] Reconstruction error: 2.6114551587852475e-07


With a little bit of linear algebra fun, you find out that a linear least squares problem of type $\min \lVert Ax - b \rVert_2$ boils down to computing the QR decomposition $A=QR$ and then solving for $Rx = Q^T b$. (Of course, we need to assume that if $A \in \mathbb{R}^{m \times n}$ that $m \geq n$ and $R$ is invertible...)

In [ ]:
%%px
b = ht.random.randn(100000,1,split=None)
Qtb = Q.T @ b
x = ht.linalg.solve_triangular(R,Qtb.resplit_(0)) 
print(f'Residual at x: {float(ht.norm(A @ x - b)/norm_A):.2e}')
print(f'Gradient of squared problem at x: {float(ht.norm(A.T @ A @ x - A.T @ b)/norm_A):.2e}')

[stdout:2] Residual at x: 3.15e-02
Gradient of squared problem at x: 5.89e-07


[stdout:3] Residual at x: 3.15e-02
Gradient of squared problem at x: 5.89e-07


[stdout:1] Residual at x: 3.15e-02
Gradient of squared problem at x: 5.89e-07


[stdout:0] Residual at x: 3.15e-02
Gradient of squared problem at x: 5.89e-07


If you want to solve a LASSO-regularized version of this linear regression problem, try out `heat.regression.Lasso`!

## Singular Value Decomposition

Let $X \in \mathbb{R}^{m \times n}$ be a matrix, e.g., given by a data set consisting of $m$ data points $\in \mathbb{R}^n$ stacked together. The so-called **singular value decomposition (SVD)** of $X$ is given by 

$$
	X = U \Sigma V^T
$$

where $U \in \mathbb{R}^{m \times r_X}$ and $V \in \mathbb{R}^{n \times r_X}$ have orthonormal columns, $\Sigma = \text{diag}(\sigma_1,...,\sigma_{r_X}) \in \mathbb{R}^{r_X \times r_X}$ is a diagonal matrix containing the so-called singular values $\sigma_1 \geq \sigma_2 \geq ... \geq \sigma_{r_X} > 0$, and $r_X \leq \min(m,n)$ denotes the rank of $X$ (i.e. the dimension of the subspace of $\mathbb{R}^m$ spanned by the columns of $X$). Since $\Sigma = U^T X V$ is diagonal, one can imagine this decomposition as finding orthogonal coordinate transformations under which $X$ looks "linear". Further information: [SVD on Wikipedia](https://en.wikipedia.org/wiki/Singular_value_decomposition)

Computing the **full** SVD in a distributed environment can be quite **expensive**; nevertheless, we have implemented it: 

In [ ]:
%%px
A = ht.random.rand(2000,2000, split=1)
U, S, VT = ht.linalg.svd(A)
print(U, '\n', S, '\n', VT)

[stderr:2] /p/home/jusers/baumann7/jureca/.venv/lib/python3.13/site-packages/heat/core/linalg/svd.py:184: UserWarning: You are performing the full SVD of a distributed matrix that is neither of tall-skinny nor short-fat shape. 
 This operation may be costly in terms of memory and compute time.
  warn(


[stderr:1] /p/home/jusers/baumann7/jureca/.venv/lib/python3.13/site-packages/heat/core/linalg/svd.py:184: UserWarning: You are performing the full SVD of a distributed matrix that is neither of tall-skinny nor short-fat shape. 
 This operation may be costly in terms of memory and compute time.
  warn(


[stderr:3] /p/home/jusers/baumann7/jureca/.venv/lib/python3.13/site-packages/heat/core/linalg/svd.py:184: UserWarning: You are performing the full SVD of a distributed matrix that is neither of tall-skinny nor short-fat shape. 
 This operation may be costly in terms of memory and compute time.
  warn(


[stderr:0] /p/home/jusers/baumann7/jureca/.venv/lib/python3.13/site-packages/heat/core/linalg/svd.py:184: UserWarning: You are performing the full SVD of a distributed matrix that is neither of tall-skinny nor short-fat shape. 
 This operation may be costly in terms of memory and compute time.
  warn(


%px:   0%|          | 0/4 [00:00<?, ?tasks/s]

[stdout:0]  
  
 


[stdout:2]  
  
 


[stdout:3]  
  
 


[stdout:1] DNDarray([[-2.2612e-02, -1.2687e-02, -4.4759e-03,  ...,  3.9940e-02, -1.0227e-02,  1.7791e-02],
          [-2.2136e-02, -1.4418e-02, -1.8957e-02,  ..., -3.6642e-03,  3.2359e-02,  5.8623e-02],
          [-2.2471e-02, -2.8934e-02,  8.2186e-03,  ..., -3.8701e-02, -2.2510e-02, -2.8751e-03],
          ...,
          [-2.2646e-02,  4.3410e-03,  2.7825e-02,  ...,  2.8433e-02,  5.7484e-02,  2.4506e-03],
          [-2.2508e-02, -4.1226e-03, -3.2982e-02,  ..., -3.2811e-02,  2.1914e-02, -4.7072e-03],
          [-2.2282e-02, -3.4762e-02, -1.8907e-02,  ..., -3.1932e-02, -1.6802e-02,  2.8414e-02]], dtype=ht.float32, device=gpu:0, split=1) 
 DNDarray([9.9995e+02, 2.5736e+01, 2.5595e+01,  ..., 1.5314e-02, 1.0256e-02, 4.9367e-03], dtype=ht.float32, device=gpu:0, split=0) 
 DNDarray([[-2.1963e-02, -2.1858e-02, -2.2373e-02,  ..., -2.2256e-02, -2.2427e-02, -2.1691e-02],
          [-4.0820e-02, -3.4550e-02, -4.9587e-03,  ..., -1.2642e-02,  1.6130e-02,  4.5199e-02],
          [ 8.7595e-03, -4.726

In [ ]:
%%px
norm_A = ht.linalg.norm(A, ord=ht.inf)
print(ht.linalg.norm(U.T @ U - ht.eye(U.shape[1]), ord=ht.inf) / norm_A)

[stdout:2] 


[stdout:0] 


[stdout:1] DNDarray([1.1337e-05], dtype=ht.float32, device=gpu:0, split=None)


[stdout:3] 


In [ ]:
%%px
print(ht.linalg.norm(VT @ VT.T - ht.eye(VT.shape[0]), ord=ht.inf) / norm_A)

[stdout:1] DNDarray([1.1336e-05], dtype=ht.float32, device=gpu:0, split=None)


[stdout:0] 


[stdout:2] 


[stdout:3] 


In [ ]:
%%px
print(ht.norm(A - U @ ht.diag(S) @  VT, ord=ht.inf)/norm_A)

[stdout:0] 


[stdout:1] DNDarray([0.0005], dtype=ht.float32, device=gpu:0, split=None)


[stdout:2] 


[stdout:3] 


If the number of rows is much higher than the number of columns (we call such matrices "tall-skinny"), a more efficient implementation of SVD is available than in the general case:

In [ ]:
%%px 
X = ht.random.rand(100000,2000, split=0)
U, S, VT = ht.linalg.svd(X)
norm_X = ht.linalg.norm(X, ord=ht.inf)
print(ht.linalg.norm(U.T @ U - ht.eye(U.shape[1]), ord=ht.inf) / norm_X)
print(ht.linalg.norm(VT @ VT.T - ht.eye(VT.shape[0]), ord=ht.inf) / norm_X)
print(ht.norm(X - U @ ht.diag(S) @  VT, ord=ht.inf) / norm_X)

[stdout:0] 




[stdout:2] 




[stdout:3] 




[stdout:1] DNDarray([3.1329e-06], dtype=ht.float32, device=gpu:0, split=None)
DNDarray([1.8098e-06], dtype=ht.float32, device=gpu:0, split=None)
DNDarray([0.0006], dtype=ht.float32, device=gpu:0, split=None)


## Truncated SVD 

So, what to do if your matrix is very large but not tall-skinny? 

Let us start with a brief background on how SVD is actually used in applications... In data science, SVD is more often known as **principle component analysis (PCA)**, the columns of $U$ being called the principle components of $X$. In fact, in many applications **truncated SVD/PCA** suffices: to reduce $X$ to the "essential" information, one chooses a truncation rank $0 < r \leq r_X$ and considers the truncated SVD/PCA given by 

$$
X \approx X_r := U_{[:,:r]} \Sigma_{[:r,:r]} V_{[:,:r]}^T
$$

where we have used `numpy`-like notation for selecting only the first $r$ columns of $U$ and $V$, respectively. The rationale behind this is that if the first $r$ singular values of $X$ are much larger than the remaining ones, $X_r$ will still contain all "essential" information contained in $X$; in mathematical terms: 

$$
\lVert X_r - X \rVert_{F}^2 = \sum_{i=r+1}^{r_X} \sigma_i^2, 
$$

where $\lVert \cdot \rVert_F$ denotes the Frobenius norm. Thus, truncated SVD/PCA may be used for, e.g.,  
* filtering away non-essential information in order to get a "feeling" for the main characteristics of your data set, 
* to detect linear (or "almost" linear) dependencies in your data, 
* to generate features for further processing of your data. 

In the following we present two options available in Heat to compute an **approximate truncated SVD for a large matrix**. 

### Randomized SVD 

The most easy option is so-called randomized SVD which allows you to specify the rank $r$ you are targeting as second positional argument. Here, we use randomized SVD with a certain number of oversamples and one power iteration in order to compute an approximation to the leading 10 singular values and vectors of the same matrix $A$ as above:   

In [ ]:
%%px 
Ur, Sr, Vr = ht.linalg.rsvd(X, 10, n_oversamples=10, power_iter=1)
print(Ur)
print(Sr)
print(Vr)

%px:   0%|          | 0/4 [00:00<?, ?tasks/s]

[stdout:0] 




[stdout:2] 




[stdout:3] 




[stdout:1] DNDarray([[-3.1636e-03,  2.0170e-03,  4.8481e-03,  ...,  2.0023e-03, -7.0219e-04,  1.8189e-03],
          [-3.1620e-03, -1.9367e-03,  1.2747e-03,  ..., -2.6003e-03,  7.1229e-04, -3.9401e-03],
          [-3.2262e-03,  3.2963e-03, -3.2350e-03,  ..., -4.1801e-03, -3.1934e-03, -1.2424e-04],
          ...,
          [-3.1939e-03,  5.8793e-04, -6.6539e-04,  ..., -2.5322e-03,  6.4239e-03, -2.7070e-03],
          [-3.2166e-03, -7.4672e-04,  4.9018e-04,  ..., -4.2101e-03, -6.9904e-03, -1.2227e-03],
          [-3.1323e-03,  1.6376e-03, -1.9667e-03,  ...,  4.1475e-03,  8.7359e-04, -8.6089e-04]], dtype=ht.float32, device=gpu:0, split=0)
DNDarray([7071.5303,   95.1160,   94.8118,   94.7587,   94.6761,   94.4659,   94.3608,   94.2008,   94.0781,   93.9981], dtype=ht.float32, device=gpu:0, split=None)
DNDarray([[-0.0224, -0.0224, -0.0223,  ..., -0.0223, -0.0224, -0.0224],
          [ 0.0300, -0.0312,  0.0529,  ..., -0.0181,  0.0193, -0.0152],
          [-0.0225,  0.0212,  0.0262,  ..., -0.

### Hierarchical SVD 

Hierarchical SVD allows a bit more of freedom. Truncation takes place either w.r.t. a fixed truncation-rank (`heat.linalg.hsvd_rank`) or w.r.t. a desired accuracy (`heat.linalg.hsvd_rtol`). In the latter case it can be ensured that it holds for the "reconstruction error": 

$$
\frac{\lVert X - U U^T X \rVert_F}{\lVert X \rVert_F} \overset{!}{\leq} \text{rtol},
$$

where $U$ denotes the approximate left-singular vectors of $X$ computed by `heat.linalg.hsvd_rtol`. 

Let's first compute the truncated SVD by setting the relative tolerance: 

In [ ]:
%%px
# compute truncated SVD w.r.t. relative tolerance 
svd_with_reltol = ht.linalg.hsvd_rtol(X,rtol=1.0e-2,compute_sv=True,silent=False)
U = svd_with_reltol[0]
reconstruction_error = (ht.linalg.norm(X - U@U.T@X, ord='fro') / ht.linalg.norm(X, ord='fro')).item()
print(f"{reconstruction_error=} relative residual: {svd_with_reltol[3].item()} rank: {U.shape[1]}")

[stdout:1] hSVD level 0...	 processes  0		1		2		3
              current ranks: 2000		2000		2000		2000
hSVD level 1...	 processes  0		2
              current ranks: 2000		2000
hSVD level 2...	 processes  0
reconstruction_error=0.0007811845862306654 relative residual: 0.027006730437278748 rank: 2000


%px:   0%|          | 0/4 [00:00<?, ?tasks/s]

[stdout:2] reconstruction_error=0.0007811845862306654 relative residual: 0.027006730437278748 rank: 2000


[stdout:0] reconstruction_error=0.0007811845862306654 relative residual: 0.027006730437278748 rank: 2000


[stdout:3] reconstruction_error=0.0007811845862306654 relative residual: 0.027006730437278748 rank: 2000


Alternatively, you can compute a truncated SVD with a fixed truncation rank:

In [ ]:
%%px
# compute truncated SVD w.r.t. a fixed truncation rank 
svd_with_rank = ht.linalg.hsvd_rank(X, maxrank=3,compute_sv=True,silent=False)
U = svd_with_rank[0]
reconstruction_error = (ht.linalg.norm(X - U@U.T@X, ord='fro') / ht.linalg.norm(X, ord='fro')).item()
print(f"{reconstruction_error=} relative residual: {svd_with_rank[3].item()} rank: {U.shape[1]}")

[stdout:1] hSVD level 0...	 processes  0		1		2		3
              current ranks: 8		8		8		8
hSVD level 1...	 processes  0
reconstruction_error=0.4995855987071991 relative residual: 0.5008995532989502 rank: 3


[stdout:3] reconstruction_error=0.4995855987071991 relative residual: 0.5008995532989502 rank: 3


[stdout:0] reconstruction_error=0.4995855987071991 relative residual: 0.5008995532989502 rank: 3


[stdout:2] reconstruction_error=0.4995855987071991 relative residual: 0.5008995532989502 rank: 3


Once we have computed the truncated SVD, we can use it to approximate the original data matrix `X` by the truncated matrix `X_r`. 

Check out the plot below to see how Heat's truncated SVD algorithm scales with the number of MPI processes and size of the dataset.

<div>
  <img src=https://github.com/helmholtz-analytics/heat/blob/main/doc/source/_static/images/hSVD_bench_rank5.png?raw=true title="hSVD rank 5" width="30%" style="float:left"/>
  <img src=https://github.com/helmholtz-analytics/heat/blob/main/doc/source/_static/images/hSVD_bench_rank50.png?raw=true title="hSVD rank 50" width="30%" style="float:center "/>
  <img src=https://github.com/helmholtz-analytics/heat/blob/main/doc/source/_static/images/hSVD_bench_rank500.png?raw=true title="HSVD rank 500" width="30%" style="float:center"/>
</div>

## Exercises 

1. Try out different split combinations `split_A=0,1,None`, `split_B=0,1,None` for matrix-matrix multiplication. What do you observe for the split of the outcome? Does every combination take the same computing time? (Whats the likely cause for this?)

2. Compare the approximate singular values computed by randomized SVD / hierarchical SVD with the reference ones computed by full SVD. What do you observe for different choices of the parameters (number of oversamples, power iterations, truncation rank, truncation tolerance)? 